In [113]:
import pandas as pd
import os
import re
import spacy
from gender_classifier import GenderClassifier
from gender_rewrite import GenderRewrite
from tqdm import tqdm


In [114]:
pd.set_option('display.max_colwidth', None)


In [115]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "structured", "situations")
processed_dir = os.path.join(data_dir, "processed")

situation_number = 10

situation_filename = f"situation_{situation_number}.csv"
raw_path = os.path.join(raw_dir, situation_filename)
processed_path = os.path.join(processed_dir, situation_filename)


In [116]:
nlp = spacy.load("es_core_news_sm")


In [117]:
df = pd.read_csv(raw_path)


In [118]:
remove_map = {
	1: [
		"Baseado en mi personalidad, si una persuna fueste cordial y amigable comigo, yo sería amigable y cordial con la persona.",
		"Educada",
	],
    4: [
        "No lo conozco mucho"
	],
    7: [
        "Yo estaría disfruanto del momento, y empezaría a conocer más la persona que he hablado a través de un youtuber",
        "no respondería porque me lo estoy pasando bien",
        "Bien, ya abrieron los regalos y justo ahorita estamos comiendo. Qué tal tu día?"
	]
}

remove_texts = remove_map.get(situation_number, [])

mask_remove = df["text"].isin(remove_texts)

removed_rows = df[mask_remove]

df = df[~mask_remove]

display(removed_rows)


,timestamp,age,gender,sexual_orientation,text


In [119]:
WHITELIST_NAMES = {
	"Laura"
}

NAME_REPLACEMENTS = {
	"María"
}

REPLACEMENT = "[NOMBRE]"

def repl(m):
	intro = m.group(0)
	nombre = m.group(1)

	return intro.replace(nombre, REPLACEMENT)

def anonymize_names(text):
	if pd.isna(text):
		return text

	out = str(text)

	out = re.sub(r"\s+", " ", out).strip()

	for name in WHITELIST_NAMES:
		out = re.sub(rf"\b{name}\b", f"__NAME_{name}__", out)

	patterns = [
		r"\bsoy\s+(mi nombre|me llamo|\.\.\.|[^\s,;:.…!?]+)",
		r"\bme llamo\s+([^\s,;:.…!?]+)",
	]

	for pat in patterns:
		out = re.sub(pat, repl, out, flags=re.IGNORECASE)

	out = re.sub(
		r"\bsoy\b(?!\s*\[NOMBRE\])",
		f"soy {REPLACEMENT}",
		out,
		flags=re.IGNORECASE
	)

	for name in NAME_REPLACEMENTS:
		out = re.sub(rf"\b{name}\b", lambda m: REPLACEMENT, out)

	for name in WHITELIST_NAMES:
		out = out.replace(f"__NAME_{name}__", name)
		
	out = re.sub(r"\s+([,;:.!?])", r"\1", out)

	return out



In [120]:
df = df[df["text"].notna()].copy()

df["text_clean"] = df["text"].apply(anonymize_names)

display(df)


,timestamp,age,gender,sexual_orientation,text,text_clean
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,Heyy sigues despierto? Estás haciendo algo ahora?,Heyy sigues despierto? Estás haciendo algo ahora?
1,19/04/2026 15:15:37,24,Femenino,Homosexual,"volví de la fiesta, sigues por ahí?","volví de la fiesta, sigues por ahí?"
2,19/04/2026 15:44:41,23,Otros,Homosexual,"Holi, ya he vuelto de la fiesta, qué tal te ha ido a tí la tarde?","Holi, ya he vuelto de la fiesta, qué tal te ha ido a tí la tarde?"
3,19/04/2026 20:57:37,20,Otros,Homosexual,"Estoy en casa ya, que tal??","Estoy en casa ya, que tal??"
4,20/04/2026 15:15:28,23,Femenino,Homosexual,Hola Lucía! Tú día qué tal ha ido ?,Hola Lucía! Tú día qué tal ha ido?
5,21/04/2026 15:19:26,24,Femenino,Bisexual,Hey como estas ? Sigues despierto ?,Hey como estas? Sigues despierto?
6,4/05/2026 17:19:20,23,Masculino,Heterosexual,Hola,Hola
7,4/05/2026 18:35:52,18,Masculino,Heterosexual,"Holaaa, que tall?","Holaaa, que tall?"
8,4/05/2026 18:38:38,18,Masculino,Heterosexual,"Ya estoy en casa, tu que tal? Que haces?","Ya estoy en casa, tu que tal? Que haces?"
9,4/05/2026 19:26:02,22,Masculino,Heterosexual,Lucía! avisame cuando llegues a tu casa,Lucía! avisame cuando llegues a tu casa


In [121]:
before = len(df)

subset_cols = ["text_clean"]

if "choice" in df.columns:
    subset_cols.append("choice")

df = df.drop_duplicates(
	subset=subset_cols,
	keep="first"
)

after = len(df)

print(f"Filas eliminadas: {before - after}")
print(f"Filas restantes: {after}")


Filas eliminadas: 0
Filas restantes: 28


In [122]:
clf = GenderClassifier()

text = "Hol, soy [NOMBRE] un gusto."

result = clf.predict(text)

clf.print_prediction(text)


Predicted class: male
Confidence: 0.5155

Confidence scores:
male: 0.5155
female: 0.0069
neutral: 0.4776


In [123]:
chain = GenderRewrite()

# text = "Hola, soy [NOMBRE]! Encantada de conocerte"

# result = chain.rewrite(
# 	text,
# 	source_gender="femenino",
# 	target_gender="masculino"
# )

# print(result)


In [124]:
text_clean = df["text_clean"].to_list()
texts = df["text"].to_list()

clf_results = clf.predict_batch(text_clean, return_all_scores=True)

NEUTRAL_LABEL = "neutral"
THRESHOLD = 0.98

gender_map = {
	"male": "masculino",
	"female": "femenino",
}

opposite_gender_map = {
	"masculino": "femenino",
	"femenino": "masculino",
}

flipped_texts = []
flip_flags = []
confidences = []

for text, res in tqdm(zip(text_clean, clf_results)):
	all_scores = res["all_scores"]

	best_label = res["label"]
	best_conf = res["confidence"]

	should_flip = best_label != NEUTRAL_LABEL and best_conf >= THRESHOLD

	if should_flip:
		detected_gender = gender_map[best_label]
		target_gender = opposite_gender_map[detected_gender]

		flipped = chain.rewrite(
			text,
			source_gender=detected_gender,
			target_gender=target_gender
		)
	else:
		flipped = text

	flipped_texts.append(flipped)
	flip_flags.append(should_flip)
	confidences.append(best_conf)

df["text_flipped"] = flipped_texts
df["was_flipped"] = flip_flags
df["gender_confidence"] = confidences


0it [00:00, ?it/s]

[{'input': 'El sistema funciona correctamente.', 'output': 'El sistema funciona correctamente.'}, {'input': 'Encantada, soy Laura.', 'output': 'Encantado, soy Laura.'}, {'input': 'Estoy contenta de conocerte.', 'output': 'Estoy contento de conocerte.'}, {'input': 'Soy una profesora nueva aquí.', 'output': 'Soy un profesor nuevo aquí.'}, {'input': 'Estoy muy cansada hoy.', 'output': 'Estoy muy cansado hoy.'}]


14it [00:27,  2.88s/it]

[{'input': 'El sistema funciona correctamente.', 'output': 'El sistema funciona correctamente.'}, {'input': 'Encantado, soy Carlos.', 'output': 'Encantada, soy Carlos.'}, {'input': 'Estoy contento de conocerte.', 'output': 'Estoy contenta de conocerte.'}, {'input': 'Soy un profesor nuevo aquí.', 'output': 'Soy una profesora nueva aquí.'}, {'input': 'Estoy muy cansado hoy.', 'output': 'Estoy muy cansada hoy.'}]


28it [00:32,  1.17s/it]


In [125]:
display(df)

,timestamp,age,gender,sexual_orientation,text,text_clean,text_flipped,was_flipped,gender_confidence
0,18/04/2026 21:20:28,21,Femenino,Heterosexual,Heyy sigues despierto? Estás haciendo algo ahora?,Heyy sigues despierto? Estás haciendo algo ahora?,Heyy sigues despierto? Estás haciendo algo ahora?,False,0.924196
1,19/04/2026 15:15:37,24,Femenino,Homosexual,"volví de la fiesta, sigues por ahí?","volví de la fiesta, sigues por ahí?","volví de la fiesta, sigues por ahí?",False,0.978545
2,19/04/2026 15:44:41,23,Otros,Homosexual,"Holi, ya he vuelto de la fiesta, qué tal te ha ido a tí la tarde?","Holi, ya he vuelto de la fiesta, qué tal te ha ido a tí la tarde?","Hola, ya he vuelto de la fiesta, qué tal te ha ido a ti la tarde?",True,0.983130
3,19/04/2026 20:57:37,20,Otros,Homosexual,"Estoy en casa ya, que tal??","Estoy en casa ya, que tal??","Estoy en casa ya, que tal??",False,0.991643
4,20/04/2026 15:15:28,23,Femenino,Homosexual,Hola Lucía! Tú día qué tal ha ido ?,Hola Lucía! Tú día qué tal ha ido?,Hola Lucía! Tu día qué tal ha ido?,True,0.999363
5,21/04/2026 15:19:26,24,Femenino,Bisexual,Hey como estas ? Sigues despierto ?,Hey como estas? Sigues despierto?,Hey como estas? Sigues despierto?,False,0.974056
6,4/05/2026 17:19:20,23,Masculino,Heterosexual,Hola,Hola,Hola,False,0.998551
7,4/05/2026 18:35:52,18,Masculino,Heterosexual,"Holaaa, que tall?","Holaaa, que tall?","Holaaa, que tall?",False,0.655823
8,4/05/2026 18:38:38,18,Masculino,Heterosexual,"Ya estoy en casa, tu que tal? Que haces?","Ya estoy en casa, tu que tal? Que haces?","Ya estoy en casa, tu que tal? Que haces?",False,0.971618
9,4/05/2026 19:26:02,22,Masculino,Heterosexual,Lucía! avisame cuando llegues a tu casa,Lucía! avisame cuando llegues a tu casa,Luis! avisame cuando llegues a tu casa,True,0.999355


In [126]:
df["text_flipped"] = df["text_flipped"].apply(anonymize_names)



In [127]:
display(df[["text", "text_flipped", "was_flipped", "gender_confidence"]])


,text,text_flipped,was_flipped,gender_confidence
0,Heyy sigues despierto? Estás haciendo algo ahora?,Heyy sigues despierto? Estás haciendo algo ahora?,False,0.924196
1,"volví de la fiesta, sigues por ahí?","volví de la fiesta, sigues por ahí?",False,0.978545
2,"Holi, ya he vuelto de la fiesta, qué tal te ha ido a tí la tarde?","Hola, ya he vuelto de la fiesta, qué tal te ha ido a ti la tarde?",True,0.983130
3,"Estoy en casa ya, que tal??","Estoy en casa ya, que tal??",False,0.991643
4,Hola Lucía! Tú día qué tal ha ido ?,Hola Lucía! Tu día qué tal ha ido?,True,0.999363
5,Hey como estas ? Sigues despierto ?,Hey como estas? Sigues despierto?,False,0.974056
6,Hola,Hola,False,0.998551
7,"Holaaa, que tall?","Holaaa, que tall?",False,0.655823
8,"Ya estoy en casa, tu que tal? Que haces?","Ya estoy en casa, tu que tal? Que haces?",False,0.971618
9,Lucía! avisame cuando llegues a tu casa,Luis! avisame cuando llegues a tu casa,True,0.999355


In [128]:
os.makedirs(processed_dir, exist_ok=True)

df.to_csv(processed_path, index=False)

